# Importer un ZIP, reprendre la collecte, retélécharger les résultats

Ce notebook remplace les versions dont la page de départ était fixée dans le code.

## Ordre des étapes
1. Lancez **Installation**.
2. Lancez **Importer le dernier ZIP** et sélectionnez votre export. Un ZIP portant l’extension `.txt` est accepté : le contenu est vérifié, sans extraire ni exécuter de fichiers.
3. Vérifiez la page de reprise affichée, puis lancez **Collecte**.
4. Si le téléchargement automatique ne fonctionne pas, lancez **Télécharger à nouveau le ZIP**. Cette cellule ne fait **aucune requête au site** et peut être réutilisée.

Avec votre dernier export : **79 pages sauvegardées, 15 tags conservés, reprise à la page 80 (`pid=1580`)**.

## Réglages
- **Plafond total par défaut : page 89**, donc 10 nouvelles pages maximum avec cet export. Vous pouvez modifier `PAGE_LIMIT` dans la cellule d’import, dans une limite de 50 pages supplémentaires par reprise.
- **Pause : 3 secondes** entre les requêtes.
- Aucun tag Character exclu ; les catégories et groupes restent inclus. Alias non fusionnés.
- Arrêt au top 20 avec contrôle de seuil, au plafond configuré, sur erreur ou interruption.

## Sauvegarde et reprise
Chaque page validée met à jour les CSV, le rapport et les noms exacts de tags disponibles dans la session Colab. À la fin ou après un arrêt, le nouveau ZIP inclut les anciens résultats et les nouveaux.

Pour continuer ultérieurement : **téléchargez le nouveau ZIP**, puis importez-le dans la cellule 2. Ne relancez pas la cellule de collecte avec l’ancien point de reprise : un garde-fou bloque cette répétition dans la session.

Les sauvegardes ne sont pas automatiquement envoyées dans Drive ni sur votre téléphone. Téléchargez-les avant de fermer/supprimer l’environnement. Une disparition de la session peut les effacer.

## Limites
Un HTTP **429** provoque un arrêt sans réessai automatique. Une reprise ou une pause de 3 secondes ne garantit pas que le blocage soit levé. Respectez l’éventuel `Retry-After` ; ne lancez pas de reprises en boucle.

Les anciens exports ne contiennent pas tous les noms des tags non-Character. Le contrôle historique de doublons peut donc être incomplet. La base peut aussi évoluer entre deux sessions : le résultat n’est pas un instantané figé.

Aucune clé API nécessaire. Aucune image ni publication téléchargée : uniquement les tableaux de tags.


In [ ]:
#@title 1. Installation
%pip -q install requests beautifulsoup4


In [ ]:
#@title 2. Importer le dernier ZIP et choisir le plafond
import io, csv, json, zipfile, re
from pathlib import Path
from google.colab import files

PAGE_LIMIT = 89 #@param {type:"integer"}
FETCH_DELAY_SECONDS = 3 #@param {type:"number"}
IMPORT_READY = False
COLLECTION_STARTED = False

def load_checkpoint(blob):
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        def read_member(name, maximum=20_000_000):
            candidates = [i for i in z.infolist() if not i.is_dir() and Path(i.filename).name == name]
            if len(candidates) != 1 or candidates[0].file_size > maximum:
                raise ValueError('Fichier absent, ambigu ou trop volumineux : ' + name)
            return z.read(candidates[0]).decode('utf-8-sig')
        report = json.loads(read_member('rapport.json', 1_000_000))
        if not isinstance(report, dict) or 'pages_read' not in report:
            raise ValueError('Rapport incompatible : il faut un export des pages HTML, pas de l’ancienne API.')
        rows = []
        for r in csv.DictReader(io.StringIO(read_member('personnages_collectes.csv'))):
            if not r.get('name') or not re.match(r'^Character\b', r.get('type',''), re.I):
                raise ValueError('Ligne de tag invalide dans le CSV.')
            count = int(r['count'])
            if count < 0:
                raise ValueError('Compteur négatif non accepté.')
            rows.append({'name':r['name'], 'count':count, 'type':r['type'], 'character':True})
        if len({r['name'] for r in rows}) != len(rows):
            raise ValueError('Tags exacts répétés dans le CSV importé.')
        pages = int(report['pages_read'])
        tags = int(report['tags_read'])
        if pages < 0 or tags != pages * 20 or int(report['character_tags_found']) != len(rows):
            raise ValueError('Compteurs du rapport et du CSV incompatibles.')
        last = report.get('lowest_count_read')
        if pages and (not isinstance(last, int) or last < 0):
            raise ValueError('Compteur de fin de page absent ou incompatible.')
        names = {r['name'] for r in rows}
        if any(Path(i.filename).name == 'seen_tags.json' for i in z.infolist()):
            extra = json.loads(read_member('seen_tags.json'))
            if not isinstance(extra, list) or not all(isinstance(n,str) for n in extra):
                raise ValueError('Historique des tags incompatible.')
            names.update(extra)
        return report, rows, names

print('Sélectionnez votre dernier export ZIP. Une extension .txt est acceptée si le contenu est bien une archive ZIP.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Importez exactement un fichier.')
blob = next(iter(uploaded.values()))
IMPORTED_REPORT, BASELINE, HISTORICAL_KNOWN_NAMES = load_checkpoint(blob)
INITIAL_PAGES = int(IMPORTED_REPORT['pages_read'])
INITIAL_TAGS = int(IMPORTED_REPORT['tags_read'])
INITIAL_LOWEST_COUNT = IMPORTED_REPORT.get('lowest_count_read')
PAGE_LIMIT = int(PAGE_LIMIT)
FETCH_DELAY_SECONDS = float(FETCH_DELAY_SECONDS)
if not INITIAL_PAGES <= PAGE_LIMIT <= INITIAL_PAGES + 50:
    raise ValueError(f'Choisissez un plafond total entre {INITIAL_PAGES} et {INITIAL_PAGES + 50}.')
if FETCH_DELAY_SECONDS < 0:
    raise ValueError('La pause ne peut pas être négative.')
MAX_ADDITIONAL_PAGES = PAGE_LIMIT - INITIAL_PAGES
IMPORT_READY = True
print(f'Import réussi : {INITIAL_PAGES} pages, {len(BASELINE)} résultats conservés.')
print(f'Prochaine page : {INITIAL_PAGES+1} | pid={INITIAL_PAGES*20} | {MAX_ADDITIONAL_PAGES} nouvelles pages au maximum.')
print(f'Pause entre les requêtes : {FETCH_DELAY_SECONDS:g} s.')
if '429' in str(IMPORTED_REPORT.get('status','')):
    print('Le dernier export s’est arrêté sur un 429. L’import ne lève pas cette limitation : ne lancez pas de reprises en boucle.')
if IMPORTED_REPORT.get('retry_after_header'):
    print('Ancienne indication Retry-After :', IMPORTED_REPORT['retry_after_header'])
    print('Respectez le délai serveur avant de lancer la collecte.')


In [ ]:
#@title 3. Reprendre la collecte
import csv
import json
import re
import time
import shutil
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import requests
from bs4 import BeautifulSoup

TARGET = 20
BASE = 'https://rule34.xxx/index.php'


def parse_rows(html):
    soup = BeautifulSoup(html, 'html.parser')
    table = None
    for candidate in soup.find_all('table'):
        for tr in candidate.find_all('tr'):
            headers = [c.get_text(' ', strip=True).lower() for c in tr.find_all(['th', 'td'], recursive=False)]
            if headers[:3] == ['posts', 'name', 'type']:
                table = candidate
                break
        if table is not None:
            break
    if table is None:
        raise ValueError('Table Posts / Name / Type introuvable : blocage ou format différent. Aucun contournement tenté.')
    rows = []
    for tr in table.find_all('tr'):
        cells = tr.find_all(['td', 'th'], recursive=False)
        if not cells or [c.get_text(' ', strip=True).lower() for c in cells[:3]] == ['posts','name','type']:
            continue
        if len(cells) < 3:
            if tr.find('td'):
                raise ValueError('Ligne de tableau inattendue : arrêt sans ignorer silencieusement des tags.')
            continue
        number = re.sub(r'[\s,\u00a0]', '', cells[0].get_text(strip=True))
        if not number.isdigit():
            raise ValueError('Compteur illisible : arrêt pour éviter un classement incomplet.')
        link = None
        for anchor in cells[1].find_all('a', href=True):
            query = parse_qs(urlparse(anchor['href']).query)
            if query.get('page') == ['post'] and query.get('s') == ['list'] and 'tags' in query:
                link = anchor
                break
        if link is None:
            raise ValueError('Nom de tag introuvable dans une ligne : arrêt.')
        # Le texte visible peut masquer des caractères Unicode distinctifs.
        # parse_qs décode le paramètre une fois ; ne pas strip/normaliser ce nom.
        name = parse_qs(urlparse(link['href']).query, keep_blank_values=True)['tags'][0]
        if name == '':
            raise ValueError('Identifiant de tag vide : arrêt.')
        typ = cells[2].get_text(' ', strip=True)
        rows.append({'name': name, 'count': int(number), 'type': typ,
                     'character': bool(re.match(r'^Character\b', typ, flags=re.I))})
    return rows


def run_export():
    started = time.monotonic()
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    folder = (Path('/content') if Path('/content').is_dir() else Path.cwd()) / ('classement_tags_html_' + stamp)
    folder.mkdir(parents=True, exist_ok=True)
    globals()['LAST_EXPORT_DIR'] = folder
    characters = [dict(r) for r in BASELINE]
    historical_character_names = set(HISTORICAL_KNOWN_NAMES) | {r['name'] for r in characters}
    seen = set()  # Noms exacts rencontrés pendant cette nouvelle session.
    pages = INITIAL_PAGES
    last_count = INITIAL_LOWEST_COUNT
    retry_after = None
    requests_sent = 0
    threshold = None
    status = f'Reprise après {INITIAL_PAGES} pages ; plafond total {PAGE_LIMIT}.'
    top20_obtained = bool(IMPORTED_REPORT.get('top20_obtained_from_sorted_prefix', False))

    def save():
        ranked = sorted(characters, key=lambda r: (-r['count'], r['name']))
        for filename, data in [('personnages_collectes.csv', ranked), ('top20_observe.csv', ranked[:TARGET])]:
            dest = folder / filename
            temp = dest.with_suffix('.tmp')
            with temp.open('w', encoding='utf-8-sig', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=['rang_observe','name','count','type'])
                writer.writeheader()
                for i, row in enumerate(data, 1):
                    writer.writerow({'rang_observe': i, 'name': row['name'], 'count': row['count'], 'type': row['type']})
            temp.replace(dest)
        rouge = next(((i, r) for i, r in enumerate(ranked, 1) if r['name'] == 'rouge_the_bat'), None)
        report = {
            'source': BASE + '?page=tags&s=list&tags=&order_by=index_count&sort=desc',
            'script_version': 'HTML-5-upload-resume-download',
            'status': status, 'top20_obtained_from_sorted_prefix': top20_obtained,
            'maximum_pages': PAGE_LIMIT, 'pages_read': pages,
            'initial_pages': INITIAL_PAGES, 'maximum_additional_pages': MAX_ADDITIONAL_PAGES,
            'additional_pages_read': pages - INITIAL_PAGES, 'requests_sent_this_run': requests_sent,
            'artificial_delay_seconds': FETCH_DELAY_SECONDS, 'retry_after_header': retry_after,
            'tags_read': INITIAL_TAGS + len(seen), 'character_tags_found': len(characters),
            'lowest_count_read': last_count,
            'elapsed_seconds': round(time.monotonic() - started, 1),
            'saved_utc': datetime.now(timezone.utc).isoformat(),
            'rouge_rank_observed': rouge[0] if rouge else None,
            'rouge_count': rouge[1]['count'] if rouge else None,
            'scope': 'Tags classés Character par le site, à partir de pid=0, alias non fusionnés.',
            'limits': ['Les pages sont lues successivement, pas depuis un instantané figé.',
                       f'Reprise d’un export de {INITIAL_PAGES} pages ; tous les tags Character sauvegardés sont conservés.',
                       'Les anciens exports ne sauvegardaient pas tous les noms de tags : le contrôle historique des doublons peut être incomplet.',
                       'Une modification de la base peut causer des omissions malgré les contrôles de tri et de doublons.',
                       'Une espèce ou un groupe peut être classé Character par le site : ce n’est pas nécessairement un individu.',
                       'Le nombre de publications n’est pas un nombre de vues ou de recherches.',
                       'Les ex aequo sont départagés par nom dans les données collectées.',
                       'Les tags visuellement identiques mais distincts par leurs caractères Unicode sont conservés séparément.',
                       'Rouge absente de cet export ne signifie pas absente du site.']
        }
        dest = folder / 'rapport.json'
        temp = dest.with_suffix('.tmp')
        temp.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
        temp.replace(dest)
        (folder / 'seen_tags.json').write_text(json.dumps(sorted(historical_character_names | seen), ensure_ascii=False), encoding='utf-8')

    print(f'REPRISE ZIP — prochaine page {INITIAL_PAGES + 1}, plafond {PAGE_LIMIT}, budget {MAX_ADDITIONAL_PAGES} pages.')
    print(f'{len(BASELINE)} résultats importés ; aucune exclusion de catégorie.')
    print(f'Pause de {FETCH_DELAY_SECONDS} secondes entre les requêtes. Arrêt sur HTTP 429, sans réessai automatique.')
    print('Lecture des tableaux de tags uniquement : aucune image, aucune clé API.')
    print(f'Arrêt : top 20 obtenu avec contrôle de seuil, page {PAGE_LIMIT}, erreur ou interruption.')
    print('Dossier de sauvegarde :', folder)
    save()
    session = requests.Session()
    session.headers['User-Agent'] = 'LocalTagStatistics/HTML-5.0'
    try:
        for offset in range(0 if top20_obtained else MAX_ADDITIONAL_PAGES):
            page = INITIAL_PAGES + offset
            pid = page * 20
            print(f'\nPage {page + 1}/{PAGE_LIMIT} — nouvelle page {offset + 1}/{MAX_ADDITIONAL_PAGES} — pid={pid}', flush=True)
            requests_sent += 1
            response = session.get(BASE, params={'page':'tags','s':'list','tags':'',
                                                'order_by':'index_count','sort':'desc','pid':pid}, timeout=(15, 45))
            if response.status_code != 200:
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    print('Délai Retry-After indiqué par le serveur :', retry_after, flush=True)
                status = f'Arrêt HTTP {response.status_code}. Pas de contournement ni de nouvelle tentative automatique.'
                break
            rows = parse_rows(response.text)
            if not rows:
                status = 'Table sans tags : fin apparente. Moins de 20 personnages peuvent avoir été trouvés.'
                top20_obtained = len(characters) >= TARGET
                break
            if len(rows) != 20:
                status = 'Pagination différente de 20 lignes : arrêt pour éviter de sauter des tags.'
                break
            counts = [r['count'] for r in rows]
            if any(a < b for a, b in zip(counts, counts[1:])) or (last_count is not None and counts[0] > last_count):
                status = 'Ordre décroissant non respecté ou base modifiée : arrêt, résultat non validé.'
                break
            names = [r['name'] for r in rows]
            if len(set(names)) != len(names) or any(n in seen or n in historical_character_names for n in names):
                status = 'Tags répétés : pagination ou base modifiée. Arrêt, résultat non validé.'
                break
            seen.update(names)
            characters.extend(r for r in rows if r['character'])
            pages += 1
            last_count = counts[-1]
            ranked = sorted(characters, key=lambda r: (-r['count'], r['name']))
            if len(ranked) >= TARGET:
                threshold = ranked[TARGET - 1]['count']
                # Lire jusqu'à dépasser strictement le compteur du 20e pour
                # ne pas tronquer les ex aequo au seuil d'une page.
                if last_count < threshold:
                    top20_obtained = True
                    status = 'Top 20 obtenu dans le préfixe trié : seuil du 20e dépassé, contrôles de tri et de doublons réussis.'
                else:
                    status = '20 personnages trouvés ; vérification des ex aequo au seuil en cours.'
            else:
                status = 'Collecte en cours ; moins de 20 tags personnages trouvés.'
            save()
            print(f'{INITIAL_TAGS + len(seen)} tags lus au total | {len(characters)} tags personnages | {time.monotonic()-started:.0f} s', flush=True)
            print('Sauvegarde intermédiaire effectuée.', flush=True)
            if top20_obtained:
                break
            if offset == MAX_ADDITIONAL_PAGES - 1:
                status = ('Plafond de pages configuré atteint : moins de 20 tags personnages trouvés.'
                          if len(characters) < TARGET else
                          'Plafond de pages configuré atteint : au moins 20 résultats, mais ex aequo au seuil non entièrement vérifiés.')
                break
            print(f'Pause de {FETCH_DELAY_SECONDS} s avant la page suivante…', flush=True)
            time.sleep(FETCH_DELAY_SECONDS)
    except KeyboardInterrupt:
        status = 'Arrêt demandé : résultats partiels sauvegardés.'
    except requests.RequestException:
        status = 'Erreur réseau ou délai dépassé : arrêt, sans nouvelle tentative automatique.'
    except ValueError as exc:
        status = str(exc)
    finally:
        session.close()
    save()
    print('\n' + status)
    print('Pages traitées :', pages, '| Personnages trouvés :', len(characters))
    print('Top 20 obtenu avec contrôles du préfixe trié :', top20_obtained)
    archive = shutil.make_archive(str(folder), 'zip', root_dir=str(folder))
    print('Résultats :', archive)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        pass
    except Exception:
        print('Utilisez le panneau Fichiers de Colab pour télécharger le ZIP.')
    return folder




if not globals().get('IMPORT_READY', False):
    raise RuntimeError('Exécutez d’abord la cellule d’import du ZIP.')
if globals().get('COLLECTION_STARTED', False):
    raise RuntimeError('Cette reprise a déjà été lancée. Téléchargez le dernier ZIP avec la cellule 4, puis importez-le avec la cellule 2 avant de reprendre.')
COLLECTION_STARTED = True
LAST_EXPORT_DIR = run_export()


In [ ]:
#@title 4. Télécharger à nouveau le ZIP — aucune requête au site
from pathlib import Path
import shutil
from google.colab import files

folder = globals().get('LAST_EXPORT_DIR')
if folder is None or not Path(folder).is_dir():
    raise RuntimeError('Aucune sauvegarde de cette collecte dans la session. Lancez la collecte ou récupérez votre ancien ZIP dans les téléchargements du téléphone.')
folder = Path(folder)
if not (folder / 'rapport.json').is_file():
    raise RuntimeError('La sauvegarde n’est pas encore disponible. Attendez la fin ou l’arrêt de la collecte.')
archive = shutil.make_archive(str(folder), 'zip', root_dir=str(folder))
print('Téléchargement des résultats sauvegardés, sans relancer la collecte :', archive)
files.download(archive)
